# ED-MRDT v1.1 — EGFR-EGF Full Analysis

Complete pipeline: **Brownian Dynamics → Virtual FLIM Microscope → Image Analysis → Causal Inference (CTE v2)**

This notebook runs N independent replicas of the EGFR-EGF system and performs:
1. Simulation (BD + photophysics + microscope)
2. Video generation (BD particles + fluorescence intensity + FLIM lifetime)
3. Conditional Transfer Entropy analysis with trial-shuffle surrogates
4. Convergence analysis

---

## 1. Setup

In [ ]:
# === OPTION 1: Local execution (project already on disk) ===
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# === OPTION 2: Google Colab (uncomment these lines) ===
# !pip install numba scipy matplotlib pillow -q
# # Upload ed_mrdt.zip or clone from GitHub:
# # !git clone https://github.com/<user>/ed_mrdt_v1.1.git
# # !unzip ed_mrdt.zip
# import sys; sys.path.insert(0, 'ed_mrdt_v1.1_PIP3-PTEN_PAPER_1')

import time
import json
import numpy as np
from IPython.display import display, Image, HTML

from ed_mrdt.config import load_config
from ed_mrdt.pipeline import SimulationPipeline
from ed_mrdt.video import generate_all_replica_videos
from l6_te_redesign import (
    sim_results_to_trials_dict,
    CausalPairSpecV2,
    compute_causal_flim_generic_v2,
    discretize_uniform,
    ensemble_conditional_te_discrete,
)

print('All imports OK')

## 2. Configuration (EDITABLE)

**Modify any parameter below** to customize the experiment. All values have comments explaining their effect.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT CONFIGURATION — EDIT FREELY                     ║
# ║                                                              ║
# ║  All biological parameters from published experimental data: ║
# ║    EGFR density: Coffman et al. (2016) PMC4836855            ║
# ║    EGFR diffusion: Schaaf et al. (2019) Biophys J 116        ║
# ║    EGF-EGFR kinetics: Schlessinger (1988) Biochemistry 27    ║
# ║    R0 Alexa488-555: Bhatt et al. (2017) Dyes Pigm 145       ║
# ╚══════════════════════════════════════════════════════════════╝

# --- Simulation parameters ---
YAML_CONFIG      = '../ed_mrdt/examples/mini_demo.yaml'
N_REPLICAS       = 20          # Independent runs
N_FRAMES         = 30          # Frames per replica
FRAME_INTERVAL   = 0.1        # Seconds between frames
BD_TIMESTEP      = 1e-4       # 100 us BD timestep

# --- Membrane & domain ---
# A431 cells: ~636 EGFR/um2 (Coffman et al. 2016, SPRi)
# Endogenous: ~100 EGFR/um2
# We model a 2x2 um2 patch at moderate expression (~200/um2)
MEMBRANE_SIZE    = [2.0, 2.0]  # um (2x2 um2 = 4 um2 membrane patch)
PIXELS           = 34          # 34x34 pixels -> 59 nm/pixel (Nyquist for confocal)
PIXEL_SIZE_NM    = 59.0        # nm/pixel

# --- Particle populations (from experimental densities) ---
AREA_UM2 = MEMBRANE_SIZE[0] * MEMBRANE_SIZE[1]  # 4 um2
OVERRIDE_POPULATIONS = {
    'EGFR': int(200 * AREA_UM2),   # 800 — moderate expression (~200/um2, Coffman 2016)
    'EGF':  int(100 * AREA_UM2),   # 400 — subsaturating ligand (~100 nM bath)
}

# --- Diffusion coefficients (override YAML) ---
# EGFR: D = 0.068 um2/s (Schaaf et al. 2019, single-particle tracking)
# EGF free: D ~ 10 um2/s (soluble ligand, YAML default OK)
# EGFR_EGF complex: D = 0.04 um2/s (dimer, slower)
OVERRIDE_DIFFUSION = {
    'EGFR':     0.068,   # um2/s, Schaaf et al. 2019 (Biophys J 116)
    'EGF':      10.0,    # um2/s, soluble ligand (same as YAML)
    'EGFR_EGF': 0.04,    # um2/s, receptor-ligand complex (dimer)
}

# --- Binding kinetics (override YAML) ---
# kon_2D = 0.05 um2/s (Schoeneberg & Noe 2013, typical receptor-ligand 2D)
# koff = 0.06 s-1 for monomers (Lax et al. 1993, PubMed 8347619)
OVERRIDE_KON     = 0.05   # um2/s
OVERRIDE_KOFF    = 0.06   # s-1 (monomer koff, Lax et al. 1993)

# --- Transfer Entropy parameters ---
N_SURROGATES     = 200
TE_N_BINS        = 3           # 3 bins — more resolution with 800+ particles
TE_CANDIDATE_LAGS = (1, 2, 5)
TE_LAG_CRITERION = 'asymmetry'
TE_DETREND       = 'none'

# --- Video parameters ---
VIDEO_FPS        = 3
VIDEO_DPI        = 120
VIDEO_FORMAT     = 'gif'

# --- FLIM parameters ---
TAU_D_NS         = 4.1         # Alexa488 unquenched lifetime (ns)
DONOR_CHANNEL    = 0

# --- Output ---
OUTPUT_DIR       = '../results_EGFR_EGF'
VIDEO_DIR        = f'{OUTPUT_DIR}/videos'

print(f'EGFR-EGF Configuration (literature-based):')
print(f'  Membrane: {MEMBRANE_SIZE[0]}x{MEMBRANE_SIZE[1]} um ({AREA_UM2} um2)')
print(f'  EGFR: {OVERRIDE_POPULATIONS["EGFR"]} ({OVERRIDE_POPULATIONS["EGFR"]/AREA_UM2:.0f}/um2, Coffman 2016)')
print(f'  EGF:  {OVERRIDE_POPULATIONS["EGF"]} ({OVERRIDE_POPULATIONS["EGF"]/AREA_UM2:.0f}/um2)')
print(f'  D_EGFR: {OVERRIDE_DIFFUSION["EGFR"]} um2/s (Schaaf 2019)')
print(f'  kon: {OVERRIDE_KON} um2/s, koff: {OVERRIDE_KOFF} s-1 (Lax 1993)')
print(f'  {N_REPLICAS} replicas x {N_FRAMES} frames = {N_REPLICAS * N_FRAMES} total frames')
print(f'  Effective TE points: {N_REPLICAS * (N_FRAMES-1)}')

## 3. Run Simulations

In [ ]:
# Load and configure with literature-based parameters
config = load_config(YAML_CONFIG)
config.scan.n_frames = N_FRAMES
config.scan.frame_interval = FRAME_INTERVAL
config.bd_timestep = BD_TIMESTEP

# Override membrane size
config.membrane.size = MEMBRANE_SIZE
config.scan.pixels_x = PIXELS
config.scan.pixels_y = PIXELS
config.scan.pixel_size = PIXEL_SIZE_NM

# Override populations (literature densities)
for sp_name, count in OVERRIDE_POPULATIONS.items():
    if sp_name in config.initial_populations:
        config.initial_populations[sp_name] = count

# Override diffusion coefficients (literature values)
for sp in config.species:
    if sp.name in OVERRIDE_DIFFUSION:
        sp.diffusion_coefficient = OVERRIDE_DIFFUSION[sp.name]

# Override binding kinetics (literature values)
for rxn in config.reactions:
    rxn.kon = OVERRIDE_KON
    rxn.koff = OVERRIDE_KOFF

config.validate()

print(f'System: {config.name}')
print(f'Species: {[s.name for s in config.species]}')
print(f'Membrane: {config.membrane.size} um')
print(f'Populations: {dict(config.initial_populations)}')
print(f'Diffusion: {", ".join(f"{sp.name}={sp.diffusion_coefficient}" for sp in config.species)}')
print(f'Kinetics: kon={config.reactions[0].kon}, koff={config.reactions[0].koff}')
print(f'Frames: {config.scan.n_frames}, interval: {config.scan.frame_interval}s')

In [ ]:
# Run all replicas
import copy

sim_results = []
t0 = time.perf_counter()

for r in range(N_REPLICAS):
    cfg = copy.deepcopy(config)
    if hasattr(cfg, 'random_seed'):
        cfg.random_seed = 1000 + r * 7
    
    pipe = SimulationPipeline(cfg, record_subframe=True)
    result = pipe.run()
    sim_results.append(result)
    
    complexes = [f.ground_truth.n_complexes for f in result.frames]
    print(f'  Replica {r:2d}: {result.total_photons_detected:5d} photons, '
          f'complexes={complexes[0]}->{complexes[-1]}')

wall_sim = time.perf_counter() - t0
print(f'\nTotal simulation time: {wall_sim:.1f}s ({wall_sim/60:.1f} min)')
print(f'Replicas completed: {len(sim_results)}')

## 4. Video Generation

Three videos per replica:
- **BD Particles**: molecular positions on membrane (blue=receptor, red=ligand, green=complex)
- **Fluorescence Intensity**: photon counts per pixel
- **FLIM Lifetime**: mean arrival time τ per pixel (ns)

In [ ]:
os.makedirs(VIDEO_DIR, exist_ok=True)

t0_vid = time.perf_counter()
video_paths = generate_all_replica_videos(
    sim_results, output_dir=VIDEO_DIR,
    prefix='egfr_egf', fmt=VIDEO_FORMAT,
    tau_d_ns=TAU_D_NS, fps=VIDEO_FPS, dpi=VIDEO_DPI)
wall_vid = time.perf_counter() - t0_vid

print(f'\n{len(video_paths)} replicas x 3 = {len(video_paths)*3} videos')
print(f'Video generation time: {wall_vid:.1f}s')
print(f'Output: {os.path.abspath(VIDEO_DIR)}')

In [ ]:
# Display sample videos (replica 0)
print('=== Replica 0: BD Particles ===')
display(Image(filename=video_paths[0]['bd_particles']))
print('\n=== Replica 0: Fluorescence Intensity ===')
display(Image(filename=video_paths[0]['intensity']))
print('\n=== Replica 0: FLIM Lifetime ===')
display(Image(filename=video_paths[0]['flim']))

## 5. Transfer Entropy v2 — Conditional CTE with Trial-Shuffle

Each replica = 1 trial. Ensemble CTE pools across trials to build the joint distribution.

In [ ]:
# Convert simulation results to trials dictionary
trials_dict = sim_results_to_trials_dict(sim_results, tau_d_ns=TAU_D_NS)

print('Series extracted:')
for key, arr in trials_dict.items():
    print(f'  {key:15s}: shape={arr.shape}, '
          f'range=[{arr.min():.4f}, {arr.max():.4f}], '
          f'mean={arr.mean():.4f}')

In [ ]:
# Define causal pairs
causal_pairs = [
    CausalPairSpecV2(
        source_key='n_complexes', target_key='tau_mean',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='n_bleached', target_key='intensity',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='msd', target_key='n_complexes',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='density', target_key='E_mean',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
    CausalPairSpecV2(
        source_key='E_mean', target_key='n_bleached',
        candidate_lags=TE_CANDIDATE_LAGS, n_bins=TE_N_BINS,
        n_surrogates=N_SURROGATES, lag_criterion=TE_LAG_CRITERION,
        detrend_method=TE_DETREND,
    ),
]

t0_te = time.perf_counter()
te_results = compute_causal_flim_generic_v2(trials_dict, causal_pairs)
wall_te = time.perf_counter() - t0_te

print(f'TE computation time: {wall_te:.1f}s')
print(f'Pairs analyzed: {len(te_results)}')

In [ ]:
# Results table
print(f'{"Pair":<35s} {"TE(X->Y)":>10s} {"TE(Y->X)":>10s} {"p-value":>10s} {"Sig?":>6s} {"Lag":>5s}')
print('-' * 76)

for res in te_results:
    spec = res.spec
    te = res.te_result
    pair_name = f'{spec.source_key} -> {spec.target_key}'
    sig = 'YES' if te.sig_xy and te.sig_xy.significant else 'no'
    p = te.sig_xy.p_value if te.sig_xy else float('nan')
    print(f'{pair_name:<35s} {te.te_xy_bits:>10.4f} {te.te_yx_bits:>10.4f} '
          f'{p:>10.4f} {sig:>6s} {te.lag:>5d}')

# ADF diagnostics
print('\nADF Diagnostics:')
for res in te_results:
    adf = res.te_result.adf_diagnostics
    pair_name = f'{res.spec.source_key} -> {res.spec.target_key}'
    if adf and adf.get('available'):
        print(f'  {pair_name}: X p={adf["x_adf_pvalue"]:.4f}, Y p={adf["y_adf_pvalue"]:.4f}')
    elif adf and adf.get('warning'):
        print(f'  {pair_name}: {adf["warning"]}')

## 6. Convergence Analysis

How does TE stabilize as we add more trials (replicas)?

In [ ]:
import matplotlib.pyplot as plt

# Convergence: TE vs number of trials for first pair
key_x = causal_pairs[0].source_key
key_y = causal_pairs[0].target_key
x_all = trials_dict[key_x]
y_all = trials_dict[key_y]

trial_counts = list(range(2, N_REPLICAS + 1))
te_values = []

for n_t in trial_counts:
    xd = np.array([discretize_uniform(t, TE_N_BINS) for t in x_all[:n_t]])
    yd = np.array([discretize_uniform(t, TE_N_BINS) for t in y_all[:n_t]])
    te_val = ensemble_conditional_te_discrete(xd, yd, lag=1)
    te_values.append(te_val)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(trial_counts, te_values, 'o-', color='#2196F3', linewidth=2, markersize=6)
ax.set_xlabel('Number of trials (replicas)', fontsize=11)
ax.set_ylabel('TE (bits)', fontsize=11)
ax.set_title(f'Convergence: TE({key_x} -> {key_y}) vs n_trials', fontsize=12)
ax.grid(True, alpha=0.3)
ax.axhline(y=te_values[-1], color='red', linestyle='--', alpha=0.5, label=f'Final: {te_values[-1]:.4f} bits')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/convergence_te.png', dpi=150)
plt.show()
print(f'Saved: {OUTPUT_DIR}/convergence_te.png')

## 7. Save All Results

In [ ]:
# Save comprehensive results to JSON
output_data = {
    'system': 'EGFR-EGF',
    'config': YAML_CONFIG,
    'n_replicas': N_REPLICAS,
    'n_frames': N_FRAMES,
    'frame_interval': FRAME_INTERVAL,
    'n_surrogates': N_SURROGATES,
    'te_n_bins': TE_N_BINS,
    'te_lag_criterion': TE_LAG_CRITERION,
    'te_detrend': TE_DETREND,
    'wall_time_sim_s': wall_sim,
    'wall_time_vid_s': wall_vid,
    'wall_time_te_s': wall_te,
    'pairs': [],
    'convergence': [{'n_trials': n, 'te_bits': v} for n, v in zip(trial_counts, te_values)],
    'series_stats': {k: {'mean': float(v.mean()), 'std': float(v.std()),
                         'min': float(v.min()), 'max': float(v.max())}
                     for k, v in trials_dict.items()},
    'videos': [{'replica': i, **{k: os.path.basename(v) for k, v in p.items()}}
               for i, p in enumerate(video_paths)],
}

for res in te_results:
    te = res.te_result
    pair_data = {
        'source': res.spec.source_key,
        'target': res.spec.target_key,
        'te_xy_bits': te.te_xy_bits,
        'te_yx_bits': te.te_yx_bits,
        'p_xy': te.sig_xy.p_value if te.sig_xy else None,
        'significant_xy': bool(te.sig_xy and te.sig_xy.significant),
        'p_yx': te.sig_yx.p_value if te.sig_yx else None,
        'significant_yx': bool(te.sig_yx and te.sig_yx.significant),
        'lag': te.lag,
        'effect_bits_xy': te.sig_xy.effect_bits if te.sig_xy else 0,
        'notes': list(te.notes),
    }
    output_data['pairs'].append(pair_data)

results_file = f'{OUTPUT_DIR}/EGFR_EGF_full_results.json'
with open(results_file, 'w') as f:
    json.dump(output_data, f, indent=2, default=str)

# Save trials dict as numpy
np.savez(f'{OUTPUT_DIR}/EGFR_EGF_trials.npz', **trials_dict)

print(f'Results saved: {results_file}')
print(f'Trials saved: {OUTPUT_DIR}/EGFR_EGF_trials.npz')
print(f'Videos: {VIDEO_DIR}/ ({len(video_paths)*3} files)')
print(f'\n=== ANALYSIS COMPLETE ===')